In [3]:
import os
import csv
import json
import time
import random
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup

csv_file = "./filtered_urls/blogs_urls.csv"
output_folder = "./database/blogs_json_db"
os.makedirs(output_folder, exist_ok=True)

# Selenium options with better stealth
options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# Initialize driver
driver = webdriver.Chrome(options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

def scrape_blog_article(url, soup):
    """Scrape blog article page"""
    article_data = {}
    article_data['url'] = url
    article_data['page_type'] = 'blog_article'

    # Meta information
    canonical = soup.find('link', rel='canonical')
    article_data['canonical_url'] = canonical.get('href') if canonical else None
    
    meta_desc = soup.find('meta', attrs={'name': 'description'})
    article_data['meta_description'] = meta_desc.get('content') if meta_desc else None
    
    # Open Graph metadata
    og_title = soup.find('meta', property='og:title')
    article_data['og_title'] = og_title.get('content') if og_title else None
    
    og_desc = soup.find('meta', property='og:description')
    article_data['og_description'] = og_desc.get('content') if og_desc else None
    
    og_image = soup.find('meta', property='og:image')
    article_data['og_image'] = og_image.get('content') if og_image else None
    
    # Twitter metadata
    twitter_title = soup.find('meta', attrs={'name': 'twitter:title'})
    article_data['twitter_title'] = twitter_title.get('content') if twitter_title else None
    
    twitter_desc = soup.find('meta', attrs={'name': 'twitter:description'})
    article_data['twitter_description'] = twitter_desc.get('content') if twitter_desc else None
    
    twitter_image = soup.find('meta', attrs={'name': 'twitter:image:src'})
    article_data['twitter_image'] = twitter_image.get('content') if twitter_image else None
    
    twitter_hashtags = soup.find('meta', attrs={'name': 'twitter:hashtags'})
    article_data['twitter_hashtags'] = twitter_hashtags.get('content').split(', ') if twitter_hashtags else []

    # Main article container
    article_container = soup.find('div', class_='blog__article-page')
    
    # Main header image
    main_img = soup.find('div', class_='blog__article-page__main-img')
    if main_img:
        img_tag = main_img.find('img')
        article_data['header_image'] = img_tag.get('data-src') if img_tag else None
    else:
        article_data['header_image'] = None

    # Article title
    title = soup.find('h1', class_='blog__article-page__title')
    article_data['title'] = title.text.strip() if title else None

    # Subtitle
    subtitle = soup.find('div', class_='blog__article-page__subtitle')
    article_data['subtitle'] = subtitle.text.strip() if subtitle else None

    # Table of Contents
    toc_nav = soup.find('nav', id='tocNav')
    toc_items = []
    if toc_nav:
        for li in toc_nav.find_all('li'):
            span = li.find('span', class_='text-link')
            if span:
                is_subsection = 'sub' in li.get('class', [])
                toc_items.append({
                    'text': span.text.strip(),
                    'is_subsection': is_subsection
                })
    article_data['table_of_contents'] = toc_items

    # Article content
    content_div = soup.find('div', class_='blog__article-page__content')
    if content_div:
        # Extract all headings and paragraphs in order
        content_sections = []
        current_section = None
        
        for elem in content_div.find_all(['h2', 'h3', 'p', 'ul', 'ol']):
            if elem.name == 'h2':
                if current_section:
                    content_sections.append(current_section)
                current_section = {
                    'heading': elem.text.strip(),
                    'level': 2,
                    'content': []
                }
            elif elem.name == 'h3':
                if current_section:
                    content_sections.append(current_section)
                current_section = {
                    'heading': elem.text.strip(),
                    'level': 3,
                    'content': []
                }
            elif elem.name == 'p':
                text = elem.text.strip()
                if text and current_section:
                    # Extract links from paragraph
                    links = []
                    for a in elem.find_all('a'):
                        links.append({
                            'text': a.text.strip(),
                            'url': a.get('href')
                        })
                    
                    current_section['content'].append({
                        'type': 'paragraph',
                        'text': text,
                        'links': links if links else None
                    })
            elif elem.name in ['ul', 'ol']:
                list_items = []
                for li in elem.find_all('li', recursive=False):
                    # Extract text and any bold elements
                    bold_text = []
                    for b in li.find_all(['b', 'strong']):
                        bold_text.append(b.text.strip())
                    
                    list_items.append({
                        'text': li.text.strip(),
                        'bold_parts': bold_text if bold_text else None
                    })
                    
                if list_items and current_section:
                    current_section['content'].append({
                        'type': 'ordered_list' if elem.name == 'ol' else 'unordered_list',
                        'items': list_items
                    })
        
        # Add the last section
        if current_section:
            content_sections.append(current_section)
            
        article_data['content_sections'] = content_sections
    else:
        article_data['content_sections'] = []

    # Author information
    author_div = soup.find('div', class_='blog__article-page__author')
    if author_div:
        author_img = author_div.find('img')
        author_link = author_div.find('a')
        author_bio = author_div.find('div', class_='blog__article-page__author__bio')
        
        # Published date
        published_date = None
        for div in author_div.find_all('div', class_='mt-3'):
            if 'PUBLISHED ON' in div.text:
                published_date = div.text.replace('PUBLISHED ON', '').strip()
                break
        
        # Written by text
        written_by = author_div.find('div', class_='blog__article-page__author__title')
        
        article_data['author'] = {
            'name': author_link.text.strip() if author_link else None,
            'profile_url': author_link.get('href') if author_link else None,
            'profile_image': author_img.get('src') if author_img else None,
            'bio': author_bio.text.strip() if author_bio else None,
            'published_date': published_date,
            'written_by_text': written_by.text.strip() if written_by else None
        }
    else:
        article_data['author'] = None

    # Related articles
    related_articles = []
    related_div = soup.find('div', class_='blog__article-page__related')
    if related_div:
        # Get the section title
        related_title = related_div.find('div', class_='related-title')
        article_data['related_articles_title'] = related_title.text.strip() if related_title else None
        
        for article_card in related_div.find_all('a', class_='article-card'):
            img = article_card.find('img')
            title_div = article_card.find('div', class_='article-card__title')
            desc = article_card.find('p')
            
            related_articles.append({
                'title': title_div.text.strip() if title_div else None,
                'description': desc.text.strip() if desc else None,
                'url': article_card.get('href') if article_card else None,
                'image': img.get('data-src') if img else None
            })
    else:
        article_data['related_articles_title'] = None
        
    article_data['related_articles'] = related_articles

    # Social sharing data
    share_rail = soup.find('ul', class_='sharerail')
    if share_rail:
        twitter_share = share_rail.find('div', class_='twitter-share')
        pinterest_share = share_rail.find('div', class_='pinterest-share')
        
        article_data['social_sharing'] = {
            'twitter_message': twitter_share.get('data-message') if twitter_share else None,
            'twitter_hashtags': twitter_share.get('data-hashtags') if twitter_share else None,
            'twitter_via': twitter_share.get('data-via') if twitter_share else None,
            'pinterest_message': pinterest_share.get('data-message') if pinterest_share else None,
            'pinterest_image': pinterest_share.get('data-image') if pinterest_share else None
        }
    else:
        article_data['social_sharing'] = None

    return article_data

# Load URLs
try:
    with open(csv_file, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        urls = [row['url'] for row in reader]
except FileNotFoundError:
    print(f"Error: Could not find {csv_file}")
    driver.quit()
    exit(1)

print(f"Found {len(urls)} URLs to scrape")

for idx, url in enumerate(urls, 1):
    try:
        print(f"\n[{idx}/{len(urls)}] Processing: {url}")
        
        driver.get(url)
        
        # Wait for page to load
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "h1"))
            )
        except:
            print(f"  ⚠ Timeout waiting for page load")
        
        # Random delay
        time.sleep(random.uniform(2, 4))
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # Scrape the blog article
        data = scrape_blog_article(url, soup)

        # Generate filename from URL
        filename = url.split('/')[-2] if url.endswith('/') else url.split('/')[-1]
        filename = filename.replace('.htm', '').replace('.html', '')
        
        # Clean filename
        filename = filename.replace("/", "_").replace(" ", "_").replace(":", "_")
        filename = filename + ".json"
        filepath = os.path.join(output_folder, filename)

        # Save to JSON
        with open(filepath, 'w', encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

        print(f"  ✓ Saved {filename}")
        print(f"    - Title: {data.get('title', 'N/A')}")
        print(f"    - Content sections: {len(data.get('content_sections', []))}")
        print(f"    - Related articles: {len(data.get('related_articles', []))}")
        print(f"    - TOC items: {len(data.get('table_of_contents', []))}")

    except Exception as e:
        print(f"  ✗ Failed for {url}: {str(e)}")
        import traceback
        traceback.print_exc()
        continue
    
    # Random delay between requests
    time.sleep(random.uniform(1, 3))

driver.quit()
print("\n✓ All done! Scraped data saved to:", output_folder)

Found 38 URLs to scrape

[1/38] Processing: https://www.partselect.com/blog/bosch-dishwasher-e15-error-code/
  ✓ Saved bosch-dishwasher-e15-error-code.json
    - Title: How to Fix the Bosch Dishwasher E15 Error Code
    - Content sections: 9
    - Related articles: 3
    - TOC items: 8

[2/38] Processing: https://www.partselect.com/blog/bosch-dishwasher-e22-error-code/
  ✓ Saved bosch-dishwasher-e22-error-code.json
    - Title: How to Quickly Fix the Bosch Dishwasher E22 Error
    - Content sections: 7
    - Related articles: 3
    - TOC items: 7

[3/38] Processing: https://www.partselect.com/blog/bosch-dishwasher-e24-error-code/
  ✓ Saved bosch-dishwasher-e24-error-code.json
    - Title: How to Easily Resolve the Bosch E24 Error Code
    - Content sections: 14
    - Related articles: 3
    - TOC items: 14

[4/38] Processing: https://www.partselect.com/blog/bosch-dishwasher-not-starting-and-red-light/
  ✓ Saved bosch-dishwasher-not-starting-and-red-light.json
    - Title: Bosch Dishwas

  ✓ Saved refrigerator-not-cooling.json
    - Title: Why is my Refrigerator Not Cooling?
    - Content sections: 7
    - Related articles: 3
    - TOC items: 7

[33/38] Processing: https://www.partselect.com/blog/refrigerator-tripping-breaker/
  ✓ Saved refrigerator-tripping-breaker.json
    - Title: Why Your Refrigerator is Tripping the Breaker
    - Content sections: 14
    - Related articles: 3
    - TOC items: 14

[34/38] Processing: https://www.partselect.com/blog/repair-or-replace-refrigerator-shelf/
  ✓ Saved repair-or-replace-refrigerator-shelf.json
    - Title: How to Easily Repair or Replace a Refrigerator Shelf
    - Content sections: 11
    - Related articles: 3
    - TOC items: 11

[35/38] Processing: https://www.partselect.com/blog/samsung-refrigerator-22E-error-code/
  ✓ Saved samsung-refrigerator-22E-error-code.json
    - Title: How to Fix a Samsung Fridge Error Code 22E
    - Content sections: 6
    - Related articles: 3
    - TOC items: 6

[36/38] Processing: https://